# Project 2 — Word embeddings and sequence models

**Question:** do pretrained word vectors and word order help classify WildGuardMix prompts?

Read and run Q1–Q6 in order. Each code cell names the question it addresses.
Each section ends with an answer based on its outputs, plus what remains uncertain.
The final section compares against saved Project 1 results.

We reuse the course embedding loader, encoders, classifier, batching and training loop.
Repeated project operations are in [`utils/p2/functions.py`](../utils/p2/functions.py).
The official questions remain in [`project.ipynb`](project.ipynb).

## Setup and experiment budget

This notebook is a **one-fold development experiment**, not completed five-fold CV.
It uses the maximum balanced sample: 23,078 prompts per class (46,156 total),
then trains on 80% and validates on 20% of that sample. Every fit uses ten epochs.

The two new score-improvement hypotheses are deliberately small:

1. keep distinct IDs for training words missing from GloVe, so they can be learned;
2. test LSTM word dropout because the earlier tuned LSTM overfit after its early peak.

Neither change guarantees an improvement. We retain the older GloVe-only vocabulary
as an explicit baseline and repeat the close pretrained-versus-random comparison.

On Snellius, run inside a GPU allocation and confirm that `Device: cuda` is printed.
Set `P2_HF_DATASETS_CACHE`, `P2_GLOVE_PATH`, and `P2_RESULTS_ROOT` when the files
live outside the repository. Keep `P2_RESULTS_ROOT` on persistent storage.
The official test is not loaded during model selection.


In [1]:
# Setup — imports, existing HF login and hardware
import json
import os
import platform
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "part2/clslib.py").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Keep the normal Hugging Face login location if notebook caches are redirected.
default_hf_home = Path(os.environ.get("HF_HOME") or
                       Path(os.environ.get("XDG_CACHE_HOME") or Path.home() / ".cache") / "huggingface")
os.environ.setdefault("HF_TOKEN_PATH", str(default_hf_home / "token"))

from datasets import load_dataset
import matplotlib.pyplot as plt
from importlib.metadata import version
from part2 import clslib, experiment
from utils.p2.functions import (
    ProjectWordTokenizer, balanced_sample, classification_scores, clean_wildguard_prompts,
    coverage_by_label, embedding_coverage, make_toy_embeddings, stratified_folds,
    nearest_words, prepare_fold, run_configuration, input_features,
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Python: 3.14.10
Platform: Linux-7.3.3-200.fc44.x86_64-x86_64-with-glibc2.43
torch: 2.15.0+cu132
MPS available: False
Device: cuda


In [2]:
# Setup — change these values to experiment
use_toy_data = False
seed = 42
examples_per_class = 23078  # maximum balanced sample: all retained unharmful prompts
num_folds = 5
fold_index = 0
embedding_dims = 50
num_epochs = 10  # Actually used for every fit below; start with a small pilot.
batch_size = 64
hidden_size = 64
dropout = 0.2
word_dropout = 0.1
vocab_mode = "train_full"
random_std = 0.1
learning_rate = 0.001
weight_decay = 0.0001
repeat_seeds = [seed, seed + 100, seed + 200, seed + 300, seed + 400]  # Initialization/batch-order check on the SAME data.
dataset_revision = "d29c47f41c8b51348b5c8e8c81c039b3132b66d1"
glove_path = Path(os.environ.get("P2_GLOVE_PATH", root / "part2/glove.2024.wikigiga.50d.zip"))
p1_results = root / "part1/results/project1/20261106T121706101445Z"
dataset_cache = Path(os.environ.get("P2_HF_DATASETS_CACHE", root / ".cache/huggingface/datasets"))
results_root = Path(os.environ.get("P2_RESULTS_ROOT", root / "part2/results"))
if os.environ.get("SLURM_CPUS_PER_TASK"):
    torch.set_num_threads(int(os.environ["SLURM_CPUS_PER_TASK"]))

if use_toy_data:
    examples_per_class, num_folds, embedding_dims = 6, 3, 12
    num_epochs, batch_size, hidden_size = 2, 4, 8

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
results_folder = results_root / ("toy" if use_toy_data else "pilot") / run_id
results_folder.mkdir(parents=True, exist_ok=False)
settings = dict(status="SMOKE_NOT_SCIENTIFIC" if use_toy_data else "DEVELOPMENT_PILOT",
                sample_seed=seed, examples_per_class=examples_per_class, num_folds=num_folds,
                fold_index=fold_index, embedding_dims=embedding_dims, epochs=num_epochs,
                batch_size=batch_size, hidden_size=hidden_size, dropout=dropout,
                word_dropout=word_dropout, vocab_mode=vocab_mode, random_std=random_std,
                learning_rate=learning_rate, weight_decay=weight_decay,
                repeat_seeds=repeat_seeds, device=device, dataset_revision=dataset_revision,
                embedding_source=str(glove_path), official_test_accessed=False,
                checkpoint_rule="last epoch; no validation checkpoint selection",
                vocabulary="all training words have distinct IDs; GloVe misses start from matched random rows",
                tokenizer="lowercase, punctuation removed, stopwords kept, Treebank",
                packages={name: version(name) for name in
                          ["numpy", "pandas", "torch", "nltk", "datasets", "scikit-learn"]})
(results_folder / "settings.json").write_text(json.dumps(settings, indent=2))
print(settings["status"], "|", num_epochs, "epochs per fit | outputs:", results_folder)

DEVELOPMENT_PILOT | 10 epochs per fit | outputs: /home/adame/code/school/ai/nlp1/project/nlp1-student/part2/results/pilot/20261119T162214049405Z


## Q1. Is the embedding appropriate for this dataset?

**Answer with:** the text/label definition, tokenization examples, token and type coverage,
coverage by label, and frequent missing tokens.

We start with the tutorial's English GloVe 2024 Wikipedia + Gigaword **50d** table.
English general text makes it a reasonable candidate; security jargon and typos need checking.
We lowercase, remove punctuation, and keep stopwords (including negation).
The helper extends the course tokenizer with resource-free Treebank tokenization.
This is an explicit approximation to GloVe's preprocessing, not a claim of an exact match.
Removing punctuation splits contractions and changes URLs/code; inspect the examples below.

In [3]:
# Q1a — load, clean, balance and split the same task as Project 1
if use_toy_data:
    raw = pd.DataFrame([
        {"prompt": f"Please water the garden plants number {i}",
         "prompt_harm_label": "unharmful", "subcategory": "synthetic"}
        for i in range(8)
    ] + [
        {"prompt": f"Synthetic security misuse request number {i}",
         "prompt_harm_label": "harmful", "subcategory": "synthetic"}
        for i in range(8)
    ] + [
        {"prompt": "excluded example", "prompt_harm_label": "harmful",
         "subcategory": " toxic_language_hate_speech "},
        {"prompt": "duplicate safe prompt", "prompt_harm_label": "unharmful",
         "subcategory": "synthetic"},
        {"prompt": "duplicate safe prompt", "prompt_harm_label": "unharmful",
         "subcategory": "synthetic"},
        {"prompt": "contradiction", "prompt_harm_label": "harmful",
         "subcategory": "synthetic"},
        {"prompt": "contradiction", "prompt_harm_label": "unharmful",
         "subcategory": "synthetic"},
    ])
else:
    dataset = load_dataset(
        "allenai/wildguardmix", "wildguardtrain", split="train",
        revision=dataset_revision, token=True,
        cache_dir=str(dataset_cache),
    )
    assert dataset.cache_files and all(
        dataset_revision in Path(item["filename"]).parts for item in dataset.cache_files
    ), "Cached revision differs from the requested revision."
    raw = dataset.select_columns(
        ["prompt", "prompt_harm_label", "subcategory"]).to_pandas()

data, data_audit = clean_wildguard_prompts(raw)
display(pd.Series(data_audit, name="data audit"))

if not use_toy_data:
    expected = {"original_rows": 86759, "excluded_category_rows": 3020,
                "eligible_rows": 83739, "retained_rows": 46692}
    assert all(data_audit[key] == value for key, value in expected.items())

sample = balanced_sample(data, examples_per_class, seed)
folds = stratified_folds(sample["label"], num_folds, seed)
train_indices, validation_indices = folds[fold_index]
train = sample.iloc[train_indices].reset_index(drop=True)
validation = sample.iloc[validation_indices].reset_index(drop=True)
print("First fold:", len(train), "training /", len(validation), "validation")
assert set(train.id).isdisjoint(validation.id)
split_table = sample[["id", "label"]].copy()
split_table["fold"] = -1
for fold, (_, indices) in enumerate(folds):
    split_table.loc[indices, "fold"] = fold
split_table.to_csv(results_folder / "folds.csv", index=False)
(results_folder / "data_audit.json").write_text(json.dumps(data_audit, indent=2))

original_rows                  86759.00000
excluded_category_rows          3020.00000
missing_label_rows                 0.00000
eligible_rows                  83739.00000
empty_prompt_rows                 14.00000
contradictory_label_rows          30.00000
contradictory_label_groups        15.00000
duplicate_rows                 37003.00000
retained_rows                  46692.00000
retained_unharmful             23078.00000
retained_harmful               23614.00000
retained_harmful_prevalence        0.50574
Name: data audit, dtype: float64

First fold: 36924 training / 9232 validation


378

In [4]:
# Q1b — load local vectors and inspect tokenization
if use_toy_data:
    embeddings = make_toy_embeddings(
        ["please", "water", "the", "garden", "plants", "synthetic", "security", "request",
         "number", "safe", "prompt", "dog", "bites", "man"], embedding_dims, seed)
else:
    if not glove_path.exists():
        raise FileNotFoundError(f"Local GloVe archive required: {glove_path}")
    embeddings = clslib.PretrainedEmbeddings(str(glove_path), num_dims=embedding_dims)
tokenizer = ProjectWordTokenizer(lowercase=True, remove_punct=True, remove_stopwords=False,
                                 vocab=embeddings.vocab)
examples = ["Don't share your MFA code.", "Explain authentication and phishing.",
            "Please help with my gardenn."]  # Safe examples, not dataset prompts.
display(pd.DataFrame({"text": examples, "tokens": [tokenizer.tokens(s) for s in examples]}))
print("Full embedding matrix:", embeddings.vectors.shape)
print("Vector array MiB:", round(embeddings.vectors.nbytes / 2**20, 1))

,text,tokens
0,Don't share your MFA code.,"[don, t, share, your, mfa, code]"
1,Explain authentication and phishing.,"[explain, authentication, and, phishing]"
2,Please help with my gardenn.,"[please, help, with, my, gardenn]"


Full embedding matrix: (1287615, 50)
Vector array MiB: 245.6


In [ ]:
# Q1c — GloVe coverage is different from model-vocabulary coverage
coverage, oov = embedding_coverage(train.prompt, tokenizer, embeddings.vocab)
label_coverage = coverage_by_label(train, tokenizer, embeddings.vocab)
prepared = prepare_fold(train, validation, embeddings, tokenizer,
                        vocab_mode=vocab_mode, random_std=random_std)
train_vocab_coverage, _ = embedding_coverage(train.prompt, tokenizer, prepared["vocab"])
validation_vocab_coverage, validation_oov = embedding_coverage(
    validation.prompt, tokenizer, prepared["vocab"])

display(pd.DataFrame({
    "training / GloVe": coverage,
    "training / model vocabulary": train_vocab_coverage,
    "validation / model vocabulary": validation_vocab_coverage,
}))
display(label_coverage.round(4))
display(oov.head(15))
assert train_vocab_coverage.token_coverage == 1.0
coverage.to_csv(results_folder / "embedding_coverage.csv")
label_coverage.to_csv(results_folder / "coverage_by_label.csv", index=False)
oov.to_csv(results_folder / "oov_counts.csv", index=False)
validation_oov.to_csv(results_folder / "validation_train_vocab_oov.csv", index=False)
print(f"GloVe covers {coverage.token_coverage:.2%} of token occurrences and "
      f"{coverage.type_coverage:.2%} of distinct training tokens.")
print("The model vocabulary represents 100% of training tokens, but this does not increase GloVe coverage.")
print("Validation-only words still map to UNK, so evaluation text never expands the vocabulary.")


## Q2. What does the embedding capture, and where does it fail?

**Answer with:** real neighbours for task vocabulary, an order-swap example,
and a controlled classification ablation in Q3.

GloVe assigns one vector per word, so it cannot give different representations to
different senses of the same word. Topic similarity can bring safe and unsafe requests
close together. Training words missing from GloVe now receive distinct trainable rows;
validation-only words still share `<unk>`. Mean/sum pooling also discards word order.
These limitations matter when intent depends on context or negation.


In [6]:
# Q2a — actual nearest neighbours; no invented similarities for missing words
probe_words = ["phishing", "fraud", "authentication", "mfa", "garden"]
display(pd.DataFrame({"word": probe_words, "in_vocabulary": [w in embeddings.vocab for w in probe_words]}))
neighbours = pd.concat([nearest_words(embeddings, word) for word in probe_words], ignore_index=True)
display(neighbours.round(4))
neighbours.to_csv(results_folder / "embedding_neighbours.csv", index=False)
print("These are distributional associations; inspect whether they express topic, syntax, or intent.")

,word,in_vocabulary
0,phishing,True
1,fraud,True
2,authentication,True
3,mfa,True
4,garden,True


,query,neighbour,cosine
0,phishing,scams,0.7580
1,phishing,scammers,0.7212
2,phishing,malware,0.7075
3,phishing,malicious,0.6956
4,phishing,astroturfing,0.6867
5,fraud,charges,0.8763
6,fraud,bribery,0.8589
7,fraud,alleged,0.8540
8,fraud,embezzlement,0.8407
9,fraud,alleging,0.8379


These are distributional associations; inspect whether they express topic, syntax, or intent.


In [7]:
# Q2b — a concrete limitation: pooled vectors lose word order
sentences = ["dog bites man", "man bites dog"]
vectors = [np.mean([embeddings.embed_word(t) for t in tokenizer.tokens(s)], axis=0)
           for s in sentences]
distance = float(np.linalg.norm(vectors[0] - vectors[1]))
print("Mean-vector distance for the order swap:", distance)
assert np.allclose(vectors[0], vectors[1], atol=1e-6)
print("Q2 answer: the same words produce the same pooled vector, although the roles changed.")
print("Task-specific evidence for pretraining comes from Q3's trainable GloVe vs trainable random comparison.")
print("We have not compared another pretrained family or embedding dimension.")

Mean-vector distance for the order swap: 3.3618550787650747e-07
Q2 answer: the same words produce the same pooled vector, although the roles changed.
Task-specific evidence for pretraining comes from Q3's trainable GloVe vs trainable random comparison.
We have not compared another pretrained family or embedding dimension.


## Q3. Which embed-and-pool choices matter?

**Answer with:** mean vs sum pooling, frozen vs trainable GloVe, trainable
GloVe vs trainable random vectors, and full training vocabulary vs the old GloVe-only vocabulary.

`train_full` is fitted only on the training fold. Every observed training word gets a
deterministic ID. GloVe matches copy pretrained vectors; GloVe misses receive distinct
random rows. Validation-only words map to `<unk>`. For a given seed, `mean_tuned` and
`mean_random` share the vocabulary, missing-word rows, downstream head and batch order;
only the available GloVe rows differ. Fully frozen systems also freeze random OOV rows.

The previous 10-epoch CUDA run used the old GloVe-only vocabulary and observed 0.8510
macro-F1 for `mean_tuned` versus 0.8452 for `mean_random` on seed 42. Those are historical
development results. The new experiment must be rerun before claiming that full vocabulary helps.


In [ ]:
# Q3a — shared full vocabulary plus one historical-vocabulary bridge
glove_only_prepared = prepare_fold(
    train, validation, embeddings, tokenizer,
    vocab_mode="glove_only_baseline", random_std=random_std)
display(pd.DataFrame([
    {"vocab_mode": prepared["vocab_mode"], "rows": len(prepared["vocab"]),
     "GloVe rows": int(prepared["pretrained_mask"].sum()),
     "random training rows": int((~prepared["pretrained_mask"]).sum())},
    {"vocab_mode": glove_only_prepared["vocab_mode"], "rows": len(glove_only_prepared["vocab"]),
     "GloVe rows": int(glove_only_prepared["pretrained_mask"].sum()),
     "random training rows": int((~glove_only_prepared["pretrained_mask"]).sum())},
]))
common = dict(hidden_size=hidden_size, classifier_width=hidden_size, dropout=dropout)
pool_configs = [
    dict(name="mean_frozen", encoder="mean", initialization="pretrained", frozen=True,
         vocab_mode="train_full", **common),
    dict(name="sum_frozen", encoder="sum", initialization="pretrained", frozen=True,
         vocab_mode="train_full", **common),
    dict(name="mean_tuned", encoder="mean", initialization="pretrained", frozen=False,
         vocab_mode="train_full", **common),
    dict(name="mean_random", encoder="mean", initialization="random", frozen=False,
         vocab_mode="train_full", **common),
    dict(name="mean_tuned_glove_only", encoder="mean", initialization="pretrained", frozen=False,
         vocab_mode="glove_only_baseline", **common),
]
fit_options = dict(epochs=num_epochs, batch_size=batch_size, learning_rate=learning_rate,
                   weight_decay=weight_decay, device=device)
score_rows, histories, prediction_tables = [], [], []


In [ ]:
# Q3b — matched pooling comparisons and real learning curves
for config in pool_configs:
    inputs = glove_only_prepared if config["vocab_mode"] == "glove_only_baseline" else prepared
    score, history, predictions = run_configuration(inputs, config, seed=seed, **fit_options)
    score_rows.append(score)
    histories.append(history)
    prediction_tables.append(predictions)
pool_results = pd.DataFrame(score_rows).set_index("name")
display(pool_results[["vocab_mode", "vocabulary_size", "macro_f1", "accuracy",
                      "harmful_recall", "benign_fpr", "fit_seconds"]].round(4))
for a, b, question in [
    ("mean_frozen", "sum_frozen", "sum versus mean"),
    ("mean_frozen", "mean_tuned", "fine-tuning GloVe"),
    ("mean_random", "mean_tuned", "pretraining at equal trainability"),
    ("mean_tuned_glove_only", "mean_tuned", "full versus GloVe-only vocabulary"),
]:
    delta = pool_results.loc[b, "macro_f1"] - pool_results.loc[a, "macro_f1"]
    print(f"Q3 observed difference ({question}): {delta:+.4f} macro-F1 (B - A).")

pool_history = pd.concat(histories, ignore_index=True)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for system in ["mean_tuned", "mean_random"]:
    curve = pool_history[pool_history.system == system]
    axes[0].plot(curve.epoch, curve.train_loss, "--", label=f"{system} train")
    axes[0].plot(curve.epoch, curve.validation_loss, marker="o", label=f"{system} validation")
    axes[1].plot(curve.epoch, curve.train_accuracy, "--", label=f"{system} train")
    axes[1].plot(curve.epoch, curve.accuracy, marker="o", label=f"{system} validation")
    axes[2].plot(curve.epoch, curve.macro_f1, marker="o", label=system)
for ax, title, ylabel in zip(axes,
    ["Loss", "Accuracy", "Validation macro-F1"], ["NLL", "Accuracy", "Macro-F1"]):
    ax.set(title=title, xlabel="Epoch", ylabel=ylabel)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(results_folder / "mean_tuned_vs_mean_random_curves.png", dpi=150)
plt.show()
print("These are fixed-budget development trajectories; the best visible epoch is not a final test score.")


## Q4. Does a recurrent encoder help?

**Answer with:** LSTM vs mean pooling, frozen vs trainable embeddings, half vs full
training data, classifier dropout and LSTM word dropout. We keep validation fixed for
the size comparison and use a nested balanced subset. Its vocabulary stays fixed to
isolate labelled-data size.

The historical CUDA run showed the tuned LSTM peaking early and then declining. We
therefore add one prespecified `word_dropout=0.1` experiment using the course LSTM's
input-word dropout. This is separate from `lstm_dropout`, which changes classifier
dropout from 0.2 to 0.4. Word dropout may reduce overfitting; it may also remove useful words.

We still score the final epoch. Choosing an epoch from this reported fold would be biased;
if five epochs looks promising, lock that budget and evaluate it on fresh folds.


In [ ]:
# Q4a — recurrent architecture, trainability, data size and two dropout types
lstm_configs = [
    dict(name="lstm_frozen", encoder="lstm", initialization="pretrained", frozen=True,
         word_dropout=0.0, **common),
    dict(name="lstm_tuned", encoder="lstm", initialization="pretrained", frozen=False,
         word_dropout=0.0, **common),
    dict(name="lstm_tuned_word_dropout", encoder="lstm", initialization="pretrained", frozen=False,
         word_dropout=word_dropout, **common),
    dict(name="lstm_half_data", encoder="lstm", initialization="pretrained", frozen=True,
         word_dropout=0.0, **common),
    dict(name="lstm_dropout", encoder="lstm", initialization="pretrained", frozen=True,
         word_dropout=0.0, **{**common, "dropout": 0.4}),
]
small_train = balanced_sample(train, max(1, len(train) // 4), seed)
positions = pd.Index(train.id).get_indexer(small_train.id)
small_prepared = {**prepared, "train": small_train,
                  "train_ids": [prepared["train_ids"][i] for i in positions]}
for config in lstm_configs:
    inputs = small_prepared if config["name"] == "lstm_half_data" else prepared
    score, history, predictions = run_configuration(inputs, config, seed=seed, **fit_options)
    score_rows.append(score)
    histories.append(history)
    prediction_tables.append(predictions)
all_results = pd.DataFrame(score_rows).set_index("name")
display(all_results.loc[["mean_frozen"] + [c["name"] for c in lstm_configs],
    ["n_train", "macro_f1", "accuracy", "harmful_recall", "benign_fpr",
     "dropout", "word_dropout", "trainable_parameters"]].round(4))
for a, b, question in [
    ("mean_frozen", "lstm_frozen", "LSTM versus pooling, frozen"),
    ("lstm_frozen", "lstm_tuned", "fine-tuning LSTM embeddings"),
    ("lstm_tuned", "lstm_tuned_word_dropout", "LSTM word dropout"),
    ("lstm_half_data", "lstm_frozen", "full versus half labelled data"),
    ("lstm_frozen", "lstm_dropout", "classifier dropout 0.4 versus 0.2"),
]:
    delta = all_results.loc[b, "macro_f1"] - all_results.loc[a, "macro_f1"]
    print(f"Q4 observed difference ({question}): {delta:+.4f} macro-F1 (B - A).")


In [ ]:
# Q4b — loss curves for architecture and the new regularizer
history_table = pd.concat(histories, ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, system in zip(axes, ["mean_frozen", "lstm_frozen"]):
    curve = history_table[history_table.system == system]
    ax.plot(curve.epoch, curve.train_loss, marker="o", label="training (with dropout)")
    ax.plot(curve.epoch, curve.validation_loss, marker="o", label="validation")
    ax.set(title=system, xlabel="Epoch", ylabel="NLL loss")
    ax.legend()
fig.tight_layout()
fig.savefig(results_folder / "learning_curves.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for system in ["lstm_tuned", "lstm_tuned_word_dropout"]:
    curve = history_table[history_table.system == system]
    axes[0].plot(curve.epoch, curve.train_loss, "--", label=f"{system} train")
    axes[0].plot(curve.epoch, curve.validation_loss, marker="o", label=f"{system} validation")
    axes[1].plot(curve.epoch, curve.macro_f1, marker="o", label=system)
axes[0].set(title="Tuned LSTM loss", xlabel="Epoch", ylabel="NLL loss")
axes[1].set(title="Tuned LSTM validation score", xlabel="Epoch", ylabel="Macro-F1")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(results_folder / "lstm_word_dropout_curves.png", dpi=150)
plt.show()

for system, curve in history_table.groupby("system"):
    curve = curve.sort_values("epoch")
    warning = ((curve.train_loss.diff() < 0) & (curve.validation_loss.diff() > 0)).sum()
    print(system, ":", int(warning), "epoch transitions with falling train / rising validation loss")
print("Training loss includes dropout and changing parameters; the curves are diagnostic.")


## Q5. How much evidence supports our choices?

**Search plan:** ten motivated configurations, changing one factor at a time within
each contrast. Then repeat the close pretrained-versus-random pooling comparison with new training seeds.
The sample and fold are fixed: this measures initialization/batch-order variation,
not dataset or fold variation. The CUDA run already showed a stable frozen-LSTM advantage across five seeds. The remaining uncertain result is the much smaller pretrained-versus-random gap.

After this pilot, choose a shortlist, lock settings and evaluate shared folds/seeds.
Report paired differences and descriptive spread. Do not divide a fold SD by the
square root of the number of folds as though they were independent measurements.
Do not select models on the official test.

In [ ]:
# Q5a — repeat fixed baseline systems; do not choose repeats from the winning score
repeat_configs = [pool_configs[2], pool_configs[3]]  # mean_tuned and mean_random
for model_seed in repeat_seeds[1:]:
    for config in repeat_configs:
        score, history, predictions = run_configuration(prepared, config, seed=model_seed, **fit_options)
        score_rows.append(score)
        histories.append(history)
        prediction_tables.append(predictions)
scores = pd.DataFrame(score_rows)
history_table = pd.concat(histories, ignore_index=True)
predictions = pd.concat(prediction_tables, ignore_index=True)
scores["fold"] = fold_index
predictions["fold"] = fold_index
variance = scores[scores.name.isin(["mean_tuned", "mean_random"])].groupby("name").agg(
    mean_f1=("macro_f1", "mean"), seed_sd=("macro_f1", "std"), runs=("seed", "nunique"))
display(variance.round(4))
paired = scores[scores.name.isin(["mean_tuned", "mean_random"])].pivot(
    index="seed", columns="name", values="macro_f1")
paired["pretrained_minus_random"] = paired.mean_tuned - paired.mean_random
display(paired.round(4))
print("Q5 answer: this tests whether the small pretrained-versus-random difference survives training seeds on one fixed fold.")
print("Further evaluation needs multiple held-out folds; a consistent pilot sign alone does not establish superiority.")

## Q6. Which text characteristics accompany errors?

**Features chosen before inspecting the new results:** token length, GloVe-OOV fraction,
training-vocabulary-OOV fraction, negation and security terms. GloVe-OOV asks whether
pretraining knows a word; training-vocabulary-OOV asks whether the model saw it during
training. Error rates and counts remain descriptive rather than causal.

These predictions come from a development fold used for comparisons, not an untouched test.
We show hashed IDs rather than raw prompts.


In [ ]:
# Q6a — join separate pretrained-coverage and unseen-word features to errors
features = pd.DataFrame([
    input_features(text, tokenizer, prepared["vocab"], embeddings.vocab)
    for text in validation.prompt
])
features["id"] = validation.id.to_numpy()
features["length_bin"] = pd.cut(features.token_length, [-1, 32, 128, np.inf],
                                  labels=["0–32", "33–128", "129+"])
for column in ["glove_oov_fraction", "train_vocab_oov_fraction"]:
    features[column + "_bin"] = pd.cut(features[column], [-1, 0, 0.05, 1],
                                         labels=["none", "0–5%", ">5%"])
errors = predictions[predictions.seed == seed].merge(features, on="id", validate="many_to_one")
errors["wrong"] = errors.gold != errors.prediction
errors["confidently_wrong"] = errors.wrong & errors.confidence.ge(0.8)
errors["false_negative"] = errors.gold.eq(1) & errors.prediction.eq(0)
errors["false_positive"] = errors.gold.eq(0) & errors.prediction.eq(1)
error_bins = []
major = errors[errors.system.isin(["mean_frozen", "lstm_frozen"])]
analysis_features = ["length_bin", "glove_oov_fraction_bin", "train_vocab_oov_fraction_bin",
                     "has_negation", "has_security_term"]
for feature in analysis_features:
    table = major.groupby(["system", feature], observed=True).agg(
        n=("wrong", "size"), error_rate=("wrong", "mean"),
        harmful_share=("gold", "mean"), confidently_wrong=("confidently_wrong", "sum")).reset_index()
    table = table.rename(columns={feature: "bin"}).assign(feature=feature)
    error_bins.append(table)
    print(feature)
    display(table.round(4))
error_bins = pd.concat(error_bins, ignore_index=True)
display(major[major.wrong].sort_values("confidence", ascending=False)[
    ["system", "id", "gold", "prediction", "confidence", "token_length",
     "glove_oov_fraction", "train_vocab_oov_fraction"]].head(10))
print("Q6 answer: the tables test proposed associations with group counts and class mix.")
print("Priorities: inspect high-error groups, negation/order and rare terms, then test targeted changes on fresh folds.")
print("Small groups and correlated features limit interpretation; these are not causal effects.")


## Final comparison — Project 2 versus Project 1

The first table is a **historical reference**: P1's repeated CV and official-test
results remain separate. P2 has no final test score yet.

The second comparison uses P1's exact same sample and fold, checked using every
hashed ID and label. P1 neural models used ten epochs; P2 uses the budget above.
Different tokenization, optimization and software versions also matter. Even with
the same split, this comparison cannot isolate pretraining by itself; Q3's matched
trainable initialization ablation is the relevant evidence for that question.

In [ ]:
# Final comparison A — read real saved P1 metrics, with their protocol
p1_available = (p1_results / "p1_cv_scores.csv").exists() and not use_toy_data
if p1_available:
    p1_settings = json.loads((p1_results / "settings.json").read_text())
    p1_summary = pd.read_csv(p1_results / "p1_summary.csv")
    p1_summary["system"] = p1_summary.model + "/" + p1_summary.representation
    p1_reference = p1_summary[["system", "macro_f1_mean", "macro_f1_std"]].copy()
    p1_reference["protocol"] = f"{p1_settings['runs']} samples × {p1_settings['folds']} folds; {p1_settings['epochs']} MLP epochs"
    test_path = p1_results / "p1_test_summary.csv"
    if test_path.exists():
        p1_reference = p1_reference.merge(pd.read_csv(test_path)[["system", "test_macro_f1"]], on="system")
    display(p1_reference.round(4))
    p1_reference.to_csv(results_folder / "p1_historical_reference.csv", index=False)
    print("P1 source:", p1_results)
else:
    print("P1 saved metrics unavailable or toy mode active; no historical scores invented.")

In [ ]:
# Final comparison B — only run a paired comparison when the protocol really matches
paired_protocol = (
    p1_available
    and p1_settings["dataset_revision"] == dataset_revision
    and p1_settings["examples_per_class"] == examples_per_class
    and p1_settings["folds"] == num_folds
    and seed in p1_settings["sample_seeds"]
)

if not p1_available:
    print("P1 files are unavailable here; no comparison was generated.")
elif not paired_protocol:
    print("Historical comparison only — these runs are not directly paired.")
    print(f"P1: {2 * p1_settings['examples_per_class']:,} sampled prompts; "
          f"P2: {2 * examples_per_class:,} sampled prompts.")
    print("Use p1_paired data if we need an exact model-to-model comparison.")
else:
    p1_run = p1_settings["sample_seeds"].index(seed)
    p1_folds = pd.read_csv(p1_results / "folds.csv")
    saved = p1_folds[p1_folds.run == p1_run].set_index("id")[["gold", "fold"]].sort_index()
    current = split_table.rename(columns={"label": "gold"}).set_index("id")[["gold", "fold"]].sort_index()
    pd.testing.assert_frame_equal(saved, current, check_dtype=False)
    p1_cv = pd.read_csv(p1_results / "p1_cv_scores.csv")
    p1_matched = p1_cv[(p1_cv.run == p1_run) & (p1_cv.fold == fold_index)].copy()
    p1_matched["system"] = "P1 / " + p1_matched.model + " / " + p1_matched.representation
    p1_matched["budget"] = p1_matched.model.map(
        lambda model: "solver convergence" if model == "logistic" else f"{p1_settings['epochs']} epochs")
    p2_matched = scores[scores.seed == seed].copy()
    p2_matched["system"] = "P2 / " + p2_matched.name
    p2_matched["budget"] = p2_matched.epochs.astype(str) + " epochs"
    p1_matched["n_train"] = len(split_table) - len(saved[saved.fold == fold_index])
    columns = ["system", "n_train", "budget", "macro_f1", "harmful_recall", "benign_fpr"]
    comparison = pd.concat([p1_matched[columns], p2_matched[columns]], ignore_index=True)
    display(comparison.round(4))
    comparison.to_csv(results_folder / "p1_p2_same_fold.csv", index=False)


In [ ]:
# Save — evidence for the blog, with no raw prompt text
scores.to_csv(results_folder / "p2_scores.csv", index=False)
history_table.to_csv(results_folder / "p2_training_history.csv", index=False)
predictions.to_csv(results_folder / "p2_validation_predictions.csv", index=False)
features.to_csv(results_folder / "p2_input_features.csv", index=False)
error_bins.to_csv(results_folder / "p2_error_bins.csv", index=False)
variance.to_csv(results_folder / "p2_seed_variance.csv")
paired.to_csv(results_folder / "p2_paired_differences.csv")
settings["configurations"] = pool_configs + lstm_configs
settings["p1_reference"] = str(p1_results) if p1_available else None
settings["completed_fits"] = len(scores)
(results_folder / "settings.json").write_text(json.dumps(settings, indent=2))
print("Saved", len(scores), "fits and their evidence to", results_folder)
print("Next research step: freeze a shortlist and evaluate paired folds; keep the official test for final evaluation.")